# Session & Cost Explorer

Query the micro-x-agent-loop memory database to explore sessions, token usage, tool calls, and costs.

**How to use this notebook:**
- Run cells with `Shift+Enter` (runs and moves to next cell)
- Or `Ctrl+Enter` (runs and stays on current cell)
- Cells share state — variables from earlier cells are available in later ones

## Setup
Connect to the SQLite database and load data into pandas DataFrames.

In [ ]:
import sqlite3
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# If you don't have these yet:
# pip install pandas matplotlib

DB_PATH = Path("..") / ".micro_x" / "memory.db"
conn = sqlite3.connect(DB_PATH)
print(f"Connected to {DB_PATH.resolve()}")

## 1. Sessions Overview
See all sessions with their model and message counts.

In [ ]:
sessions = pd.read_sql_query("""
    SELECT
        s.id,
        json_extract(s.metadata_json, '$.title') AS title,
        s.model,
        s.status,
        s.created_at,
        s.updated_at,
        COUNT(m.id) AS message_count
    FROM sessions s
    LEFT JOIN messages m ON m.session_id = s.id
    GROUP BY s.id
    ORDER BY s.created_at DESC
""", conn)

sessions["created_at"] = pd.to_datetime(sessions["created_at"])
sessions["updated_at"] = pd.to_datetime(sessions["updated_at"])

print(f"{len(sessions)} sessions")
sessions.head(10)

## 2. Sessions by Model
Which models have you used the most?

In [ ]:
model_counts = sessions["model"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
model_counts.plot(kind="barh", ax=ax)
ax.set_xlabel("Number of sessions")
ax.set_title("Sessions by Model")
plt.tight_layout()
plt.show()

## 3. Activity Over Time
Sessions created per day.

In [ ]:
daily = sessions.set_index("created_at").resample("D")["id"].count()

fig, ax = plt.subplots(figsize=(10, 3))
daily.plot(kind="bar", ax=ax)
ax.set_ylabel("Sessions")
ax.set_title("Daily Session Activity")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Tool Usage
Which tools are called most often?

In [ ]:
tool_calls = pd.read_sql_query("""
    SELECT tool_name, COUNT(*) AS calls, SUM(is_error) AS errors
    FROM tool_calls
    GROUP BY tool_name
    ORDER BY calls DESC
""", conn)

tool_calls["error_rate"] = (tool_calls["errors"] / tool_calls["calls"] * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
tool_calls.set_index("tool_name")["calls"].plot(kind="barh", ax=ax)
ax.set_xlabel("Total calls")
ax.set_title("Tool Usage")
plt.tight_layout()
plt.show()

tool_calls

## 5. Token Usage from Events
Parse `api_call` events to see token consumption and estimated costs.

In [ ]:
events_df = pd.read_sql_query("""
    SELECT session_id, type, payload_json, created_at
    FROM events
    WHERE type = 'metric'
    ORDER BY created_at
""", conn)

# Parse the JSON payloads
records = []
for _, row in events_df.iterrows():
    try:
        payload = json.loads(row["payload_json"])
        if payload.get("type") == "api_call":
            records.append({
                "session_id": row["session_id"],
                "timestamp": row["created_at"],
                "model": payload.get("model", ""),
                "input_tokens": payload.get("input_tokens", 0),
                "output_tokens": payload.get("output_tokens", 0),
                "cache_read_tokens": payload.get("cache_read_input_tokens", 0),
                "cache_create_tokens": payload.get("cache_creation_input_tokens", 0),
                "cost_usd": payload.get("estimated_cost_usd", 0),
                "duration_ms": payload.get("duration_ms", 0),
            })
    except (json.JSONDecodeError, TypeError):
        continue

api_calls = pd.DataFrame(records)
print(f"{len(api_calls)} API calls found")

if not api_calls.empty:
    api_calls["timestamp"] = pd.to_datetime(api_calls["timestamp"])
    api_calls.head()

In [ ]:
if not api_calls.empty:
    # Cost per session (top 15)
    session_costs = api_calls.groupby("session_id").agg(
        total_cost=('cost_usd', 'sum'),
        total_input=('input_tokens', 'sum'),
        total_output=('output_tokens', 'sum'),
        api_calls_count=('cost_usd', 'count'),
    ).sort_values("total_cost", ascending=False)

    print(f"Total spend across all sessions: ${session_costs['total_cost'].sum():.4f}")
    print(f"Average cost per session: ${session_costs['total_cost'].mean():.4f}")
    print()

    fig, ax = plt.subplots(figsize=(10, 5))
    session_costs.head(15)["total_cost"].plot(kind="bar", ax=ax)
    ax.set_ylabel("Cost (USD)")
    ax.set_title("Top 15 Most Expensive Sessions")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No API call events found — cost tracking may not be enabled.")

In [ ]:
if not api_calls.empty:
    # Cost by model
    model_costs = api_calls.groupby("model").agg(
        total_cost=('cost_usd', 'sum'),
        total_tokens=('input_tokens', lambda x: x.sum() + api_calls.loc[x.index, 'output_tokens'].sum()),
        calls=('cost_usd', 'count'),
    ).sort_values("total_cost", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    model_costs["total_cost"].plot(kind="pie", ax=axes[0], autopct="%1.1f%%")
    axes[0].set_title("Cost by Model")
    axes[0].set_ylabel("")

    model_costs["calls"].plot(kind="pie", ax=axes[1], autopct="%1.1f%%")
    axes[1].set_title("API Calls by Model")
    axes[1].set_ylabel("")
    plt.tight_layout()
    plt.show()

    model_costs

## 6. Cache Effectiveness
How well is prompt caching working?

In [ ]:
if not api_calls.empty:
    total_input = api_calls["input_tokens"].sum()
    total_cache_read = api_calls["cache_read_tokens"].sum()
    total_cache_create = api_calls["cache_create_tokens"].sum()

    print(f"Total input tokens:         {total_input:>12,}")
    print(f"Cache read tokens:          {total_cache_read:>12,}")
    print(f"Cache creation tokens:      {total_cache_create:>12,}")
    if total_input + total_cache_read > 0:
        hit_rate = total_cache_read / (total_input + total_cache_read) * 100
        print(f"Cache hit rate:             {hit_rate:>11.1f}%")

    # Cache usage over time
    daily_cache = api_calls.set_index("timestamp").resample("D").agg({
        "input_tokens": "sum",
        "cache_read_tokens": "sum",
    })

    if not daily_cache.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        daily_cache.plot(kind="bar", stacked=True, ax=ax)
        ax.set_ylabel("Tokens")
        ax.set_title("Daily Input vs Cache Read Tokens")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()

## 7. Explore a Single Session
Change the `session_id` below to drill into a specific session.

In [ ]:
# Pick the most recent session (or paste a specific ID)
session_id = sessions.iloc[0]["id"]
print(f"Exploring session: {session_id}")
print(f"Title: {sessions.iloc[0]['title']}")
print(f"Model: {sessions.iloc[0]['model']}")
print()

msgs = pd.read_sql_query("""
    SELECT seq, role, substr(content_json, 1, 200) AS content_preview, token_estimate, created_at
    FROM messages
    WHERE session_id = ?
    ORDER BY seq
""", conn, params=[session_id])

print(f"{len(msgs)} messages")
msgs

In [ ]:
# Tool calls for this session
session_tools = pd.read_sql_query("""
    SELECT tool_name, substr(input_json, 1, 100) AS input_preview,
           is_error, created_at
    FROM tool_calls
    WHERE session_id = ?
    ORDER BY created_at
""", conn, params=[session_id])

print(f"{len(session_tools)} tool calls")
session_tools

## Cleanup

In [ ]:
conn.close()
print("Database connection closed.")